# Stage 5 — Paganin + FBP (ablation copy of `steps15.py` stage 5)

Per angle: flat-field = `prb x code window` propagated together; blur it and the measurement; divide; pad; Paganin. Then one global background subtraction and an FBP.

Paths are hardcoded. `BIN` / `STRIDE` shrink the data so it fits on tomo5 (only `BIN=0` is the production pipeline). Not the production path: keep `steps15.py` in sync by hand when something here wins.

In [ ]:
# Paths (hardcoded: this notebook does not read config_steps15.conf)
PATH_H5    = '/data3/cekmekci/AtomiumS2_coded/ht/codedHT.h5'
PATH_H5_CA = '/data3/cekmekci/AtomiumS2_coded/nfp/data_nfp.h5'
PATH_CODE  = '/data3/cekmekci/AtomiumS2_coded/nfp/code.npy'
PATH_PRB   = '/data3/cekmekci/AtomiumS2_coded/nfp/prb.npy'
OUT_DIR    = '/data3/cekmekci/AtomiumS2_coded/ht'

# Size: BIN=0 is the production pipeline (2048 px, too big for tomo5)
BIN, STRIDE = 3, 8                  # 2**BIN downsample; every STRIDEth of the 7200 angles

# Knobs (production values, config_steps15.conf)
NOBJ_FACTOR = 1.25                  # object grid = detector * 1.25
DELTA_BETA  = 120.0                  # Paganin delta/beta; imag part of the object = real / DELTA_BETA
MASK_R      = 0.9                   # FBP circular mask, a SQUARED radius (tomo.py:62): 0.9 -> r = 0.949
ROT_AXIS_OFFSET_PX = (0.0, -24.0)   # (row, col), unbinned px

BG_BOX = (0.05, 0.15, 0.05, 0.15)   # (row0, row1, col0, col1), fractions of the detector. The median of the ratio in this
                                    # box (angle 0) is the background level the margin ramps to (yellow box on the plot)

PAGANIN_BG_BOX = (0.15, 0.25, 0.15, 0.25)  # (row0, row1, col0, col1), fractions of the PADDED frame. Its median on the Paganin phase
                                    # (angle 0) is subtracted from every angle (his :1231, :1263). Default = his: the top-left corner
                                    # (lime box on the plots)

CODE_SCALE = 1.0                # the code window is stretched by this factor about the detector centre before the flat field is
                                    # formed. Measured by the z1c sweep below: z1c(HT)/z1c(NFP) = 17.7599/17.75 (a ratio, not an absolute z1c).
                                    # 1.0 = off. Not in config_steps15.conf yet.

EDGE_CROP = 36                      # unbinned px stripped off every side of the division before padding: the rim where the
                                    # modelled flat-field is unreliable (measured: ~16-32 px). 0 = keep everything

TAG = f'bin{BIN}_stride{STRIDE}_nobj{NOBJ_FACTOR:g}_db{DELTA_BETA:g}_mask{MASK_R:g}_crop{EDGE_CROP:g}_pbox' + '-'.join(f'{v:.3g}' for v in PAGANIN_BG_BOX) + f'_cs{CODE_SCALE:g}'


In [ ]:
import os
import numpy as np
import cupy as cp
import cupyx.scipy.ndimage as ndimage
import h5py
import matplotlib.pyplot as plt
%matplotlib inline
from holotomocupy.propagation import Propagation

nb = 2 ** BIN
with h5py.File(PATH_H5, 'r') as f:
    energy = float(f['/exchange/energy'][0])
    det_px = float(f['/exchange/detector_pixelsize'][0])
    fdd    = float(f['/exchange/focusdetectordistance'][0])
    z1     = float(f['/exchange/z1'][0])
    ntheta_full, nz_full, n_full = f['/exchange/pdata'].shape
    theta_deg_all = f['/exchange/theta'][:].astype('float32')
    shifts_code   = f['/exchange/shifts_code'][:].astype('float32')
    shifts_cor    = f['/exchange/shifts_cor'][:].astype('float32')
with h5py.File(PATH_H5_CA, 'r') as f:
    z1_code = float(f['/exchange/z1'][0])           # nominal source-to-code distance

theta_ids = np.arange(0, ntheta_full, STRIDE)
ntheta    = len(theta_ids)
theta_deg = theta_deg_all[theta_ids]
theta_rad = np.deg2rad(theta_deg)

nz, n   = nz_full // nb, n_full // nb               # binned detector
nobj    = round(n * NOBJ_FACTOR)                    # object grid
nzobj   = round(nz * NOBJ_FACTOR)
pad8    = nobj // 8                                 # reflect pad around multiPaganin (his)
margin  = (nobj - n) // 2                           # the real data sits at [margin, margin + n) in x
marginz = (nzobj - nz) // 2                         #                                       in z

pos_obj = (shifts_cor + np.array(ROT_AXIS_OFFSET_PX, dtype='float32')) / nb     # binned px

wavelength     = 1.24e-9 / energy
z2             = fdd - z1
magnification  = fdd / z1
distance       = z1 * z2 / fdd
voxelsize_full = det_px / magnification             # unbinned
voxelsize      = voxelsize_full * nb

z2c                     = z1 - z1_code
magnification_code      = z1 / z1_code
distance_code           = (z1_code * z2c) / (z1_code + z2c)
norm_magnification_code = magnification_code / (z1_code / z1)
distance_code           = distance_code * norm_magnification_code**2 * (z1_code / z1)**2

print(f'{ntheta} angles (stride {STRIDE}), detector {nz}x{n} (bin {nb}), object {nzobj}x{nobj}, margin {margin} px, '
      f'rec_paganin ~ {ntheta * nzobj * nobj * 4 / 1e9:.2f} GB')


In [ ]:
# Flat-field = what the detector records with no sample, modelled as the solver does (RecCoded.F2):
# window the code over the detector at this angle, multiply by the probe, propagate that ONE field
# code plane -> sample -> detector.
code_gpu = cp.asarray(np.load(PATH_CODE).astype('complex64'))      # 8192^2
prb_gpu  = cp.asarray(np.load(PATH_PRB).astype('complex64'))
ncode_full = code_gpu.shape[0]

cl_prop      = Propagation(n_full, nz_full, 1, 1, wavelength, voxelsize_full, [distance])
cl_prop_code = Propagation(n_full, nz_full, 1, 1, wavelength, voxelsize_full, [distance_code])


def rescale_about_centre(w, scale):
    """Stretch the 2D complex window w by `scale` about its centre (cubic spline): out(u) = w(c + (u - c) * scale).
    Same sampling as the z1c sweep cells below (there scale = z / Z_NFP)."""
    if scale == 1.0:
        return w
    h, wd = w.shape
    co = cp.stack(cp.meshgrid((cp.arange(h, dtype='float32') - h / 2) * scale + h / 2,
                              (cp.arange(wd, dtype='float32') - wd / 2) * scale + wd / 2, indexing='ij'))
    return (ndimage.map_coordinates(w.real.copy(), co, order=3, mode='nearest')
            + 1j * ndimage.map_coordinates(w.imag.copy(), co, order=3, mode='nearest')).astype('complex64')


def code_window(k, G=8):
    """Code over the detector at angle k: integer crop from shifts_code, sub-pixel remainder by
    Fourier shift (G px guard band so nothing wraps in), then stretch by CODE_SCALE about the window
    centre (the HT code plane sits slightly off the NFP one). fourier_shift_2d is in the next cell."""
    dy, dx = int(round(shifts_code[k, 0])), int(round(shifts_code[k, 1]))
    cr, cc = ncode_full // 2 - dy, ncode_full // 2 - dx
    w = code_gpu[cr - nz_full // 2 - G: cr + nz_full // 2 + G, cc - n_full // 2 - G: cc + n_full // 2 + G]
    w = fourier_shift_2d(w, dc=float(shifts_code[k, 1] - dx), dr=float(shifts_code[k, 0] - dy))
    return rescale_about_centre(w, CODE_SCALE)[G:-G, G:-G]            # stretch about the window centre, then drop the guard


def flat_field(k):
    return cp.abs(cl_prop.D(cl_prop_code.D(prb_gpu * code_window(k), 0), 0)) ** 2


In [ ]:
def multiPaganin(data, dist, wavelength, voxelsize, delta_beta, alpha):
    fy = cp.fft.fftfreq(data.shape[-2], d=voxelsize).astype('float32')
    fx = cp.fft.fftfreq(data.shape[-1], d=voxelsize).astype('float32')
    fx, fy = cp.meshgrid(fx, fy)
    rad_freq = cp.fft.fft2(data)
    taylorExp = 1 + wavelength * dist * cp.pi * delta_beta * (fx**2 + fy**2)
    numerator = taylorExp * rad_freq
    denominator = taylorExp**2 + alpha
    phase = cp.log(cp.real(cp.fft.ifft2(numerator / denominator)))
    return delta_beta * 0.5 * phase


def fourier_shift_2d(arr, dc, dr):
    """Sub-pixel shift. Real in -> real out; complex in -> complex out."""
    nr, nc = arr.shape
    fr = cp.fft.fftfreq(nr).astype('float32')
    fc = cp.fft.fftfreq(nc).astype('float32')
    Fc, Fr = cp.meshgrid(fc, fr)
    ramp = cp.exp(-2j * cp.pi * (Fr * dr + Fc * dc)).astype('complex64')
    out = cp.fft.ifft2(cp.fft.fft2(arr.astype('complex64')) * ramp)
    return out if arr.dtype.kind == 'c' else out.real.astype('float32')


def bin2(a, nb):
    """Box-average an (h, w) array down by nb in both axes."""
    if nb == 1:
        return a
    h, w = a.shape
    return a.reshape(h // nb, nb, w // nb, nb).mean(axis=(1, 3))


def pad_taper(proj, nzobj, nobj, bg):
    """Linear ramp from the edge to `bg` across the whole margin, in both axes.
    His steps15.py:916-919 ramps x only and replicates the edge in y; our vertical edges carry
    structure (the sample mount leaves the frame) and replicated rows are the same at every angle,
    which FBP back-projects onto the axis (measured earlier: a blob ~4x the sample contrast)."""
    nz_, n_ = proj.shape
    py0, px0 = (nzobj - nz_) // 2, (nobj - n_) // 2
    return cp.pad(proj, ((py0, nzobj - nz_ - py0), (px0, nobj - n_ - px0)), 'linear_ramp', end_values=bg)


In [ ]:
VIZ_K = [0, ntheta // 4, ntheta // 2, 3 * ntheta // 4]      # angles kept for the sanity plots at the end
viz   = {}

# his steps15.py:1144-1147: FWHM 17 unbinned px, applied at full resolution before binning
sigma = 17.0 / (2 * np.sqrt(2 * np.log(2)))

r0, r1, c0, c1 = BG_BOX
crop = round(EDGE_CROP / nb)                                      # binned px stripped off each side
bg_box = (int(r0 * nz), int(r1 * nz), int(c0 * n), int(c1 * n))      # rows / cols of the binned detector
if bg_box[0] < crop or bg_box[2] < crop or bg_box[1] > nz - crop or bg_box[3] > n - crop:
    print('WARNING: BG_BOX overlaps the stripped rim (EDGE_CROP); move it inside the kept region')

rec_paganin = np.zeros((ntheta, nzobj, nobj), dtype='float32')
with h5py.File(PATH_H5, 'r') as f:
    pdata = f['/exchange/pdata']
    for i, k in enumerate(theta_ids):
        frame = cp.asarray(pdata[k].astype('float32'))                                  # measurement
        flat  = flat_field(k)                                                           # prb x code, propagated together
        num = ndimage.gaussian_filter(frame, sigma)
        den = ndimage.gaussian_filter(flat, sigma)
        proj = bin2(num / (den + 1e-5), nb)                                             # his epsilon
        if i == 0:                                                                      # pad level: measured once, from angle 0
            bg = float(cp.mean(proj[bg_box[0]:bg_box[1], bg_box[2]:bg_box[3]]))
            print(f'pad level = median of BG_BOX in angle 0 = {bg:.4f}')

        kept = proj[crop:nz - crop, crop:n - crop]                                      # strip the unreliable rim, pad the rest
        proj_padded = pad_taper(kept, nzobj, nobj, bg=bg)
        p = cp.pad(proj_padded, pad8, 'reflect')                                        # his: against FFT wrap-around
        phase = multiPaganin(p, distance, wavelength, voxelsize, DELTA_BETA, 0.01)[pad8:-pad8, pad8:-pad8]
        phase = fourier_shift_2d(phase, dc=-pos_obj[k, 1], dr=-pos_obj[k, 0])

        if i in VIZ_K:
            viz[i] = {key: x.get() for key, x in dict(flat=flat, flat_blur=den, frame=frame, frame_blur=num,
                                                      ratio=proj, padded=proj_padded, phase=phase).items()}
            viz[i]['theta'] = float(theta_deg[i])
        rec_paganin[i] = phase.get()
        if i % 200 == 0:
            print(f'  {i:5d}/{ntheta}  phase [{rec_paganin[i].min():.4f}, {rec_paganin[i].max():.4f}]', flush=True)
cp.get_default_memory_pool().free_all_blocks()

# One global background off the phase (his :1231, :1263): median of PAGANIN_BG_BOX on the Paganin phase, angle 0
pbox = tuple(int(f * s) for f, s in zip(PAGANIN_BG_BOX, (nzobj, nzobj, nobj, nobj)))       # rows / cols of the padded frame
assert pbox[1] > pbox[0] and pbox[3] > pbox[2], 'PAGANIN_BG_BOX is empty'
global_bg = float(np.mean(rec_paganin[0, pbox[0]:pbox[1], pbox[2]:pbox[3]]))
rec_paganin -= global_bg
print(f'global_bg {global_bg:.4f} subtracted;  rec_paganin {rec_paganin.shape}  [{rec_paganin.min():.4f}, {rec_paganin.max():.4f}]')


In [ ]:
from holotomocupy.tomo import Tomo

# Real-first: FBP the real phase once (float32), then imag = real / DELTA_BETA (his :1304).
# A complex sinogram makes Tomo.RT put the real channel's spurious imaginary part (~0.4% of Re) into Im,
# which is DELTA_BETA times weaker: the fine vertical lines. float32 input drops it (diagnostic below).
cl = Tomo(nobj, 8, theta_rad, mask_r=MASK_R)
obj_fbp = np.zeros((nzobj, nobj, nobj), dtype='complex64')
for z0 in range(0, nzobj, 8):
    L = cl.fbp(cp.asarray(rec_paganin[:, z0:z0 + 8, :]), 'ramp').get()
    obj_fbp[z0:z0 + 8].real = L
    obj_fbp[z0:z0 + 8].imag = L / DELTA_BETA
cp.get_default_memory_pool().free_all_blocks()
print(f'obj_fbp {obj_fbp.shape}  real [{obj_fbp.real.min():.4f}, {obj_fbp.real.max():.4f}]'
      f'  imag [{obj_fbp.imag.min():.4f}, {obj_fbp.imag.max():.4f}]')


In [ ]:
# Ablation output, never the production obj_init.npy -- that one is written
# by steps15.py at full size.
out = os.path.join(OUT_DIR, f'obj_init_ablation_{TAG}.npy')
np.save(out, obj_fbp)
print('saved', out, f'({obj_fbp.nbytes / 1e9:.2f} GB)')

## Sanity plots

**Detector frame** (first figure): flat-field, blurred flat-field, measured frame, blurred frame and the division, for 4 angles. Axes are fractions of the detector. **Yellow** = `BG_BOX`; its median in the division at angle 0 is the pad level.

**Padded frame** (second figure): the division going into Paganin, the Paganin phase before and after the global background subtraction, and profiles through the padded division. **Red dashed** = where the detector sits, **yellow** = `BG_BOX` moved into padded coordinates, **lime** = `PAGANIN_BG_BOX`, where `global_bg` is measured on the Paganin phase.

In [ ]:
# DETECTOR frame: every panel is in fractions of the detector, so the yellow BG_BOX sits in the same place on all of them.
ks = sorted(viz)


def box(a, x, y, w, h, color, ls='-'):
    """Rectangle with a dark outline, so it shows on bright and dark backgrounds alike."""
    a.add_patch(plt.Rectangle((x, y), w, h, fill=False, ec='k', lw=3.5))
    a.add_patch(plt.Rectangle((x, y), w, h, fill=False, ec=color, lw=1.5, ls=ls))


rows = [('flat-field  |prop(prb x code)|^2', 'flat'), ('flat-field, blurred', 'flat_blur'), ('measured frame', 'frame'),
        ('frame, blurred', 'frame_blur'), ('division: blurred frame / blurred flat-field', 'ratio')]
fig, ax = plt.subplots(len(rows), len(ks), figsize=(4.2 * len(ks), 4 * len(rows)), squeeze=False)
for r, (name, key) in enumerate(rows):
    for c, i in enumerate(ks):
        img = viz[i][key]; a = ax[r][c]
        im = a.imshow(img, cmap='gray', vmin=np.percentile(img, 1), vmax=np.percentile(img, 99), extent=(0, 1, 1, 0))
        box(a, c0, r0, c1 - c0, r1 - r0, 'yellow')
        box(a, crop / n, crop / nz, 1 - 2 * crop / n, 1 - 2 * crop / nz, 'cyan', '--')           # kept after the strip
        a.set_title(f"{name}\nangle {viz[i]['theta']:.1f} deg", fontsize=9); plt.colorbar(im, ax=a, fraction=0.046)
plt.suptitle(f'{TAG}\nyellow = BG_BOX, cyan dashed = kept after stripping EDGE_CROP={EDGE_CROP:g} px; BG_BOX median in the division at angle {viz[ks[0]]["theta"]:.1f} deg = pad level {bg:.4f}', fontsize=10)
plt.tight_layout(); plt.show()


In [ ]:
# PADDED frame (the Paganin grid): the data sits in the red dashed rectangle, the margin is the ramp.
fig, ax = plt.subplots(4, len(ks), figsize=(4.2 * len(ks), 16.5), squeeze=False)
for c, i in enumerate(ks):
    for r, (name, img) in enumerate((('1) padded division, into Paganin', viz[i]['padded']),
                                     ('2) Paganin, before subtraction', viz[i]['phase']),
                                     ('3) Paganin - global_bg', rec_paganin[i]))):
        a = ax[r][c]
        im = a.imshow(img, cmap='gray', vmin=np.percentile(img, 1), vmax=np.percentile(img, 99))
        a.add_patch(plt.Rectangle((margin + crop, marginz + crop), n - 2 * crop, nz - 2 * crop, fill=False, ec='r', lw=0.8, ls='--'))   # kept data
        box(a, pbox[2], pbox[0], pbox[3] - pbox[2], pbox[1] - pbox[0], 'lime')                          # PAGANIN_BG_BOX
        med = '' if r == 0 else f'   (lime-box median {np.median(img[pbox[0]:pbox[1], pbox[2]:pbox[3]]):+.3f})'
        a.set_title(f"{name}\nangle {viz[i]['theta']:.1f} deg{med}", fontsize=9); a.axis('off'); plt.colorbar(im, ax=a, fraction=0.046)
    box(ax[0][c], margin + bg_box[2], marginz + bg_box[0], bg_box[3] - bg_box[2], bg_box[1] - bg_box[0], 'yellow')   # BG_BOX in padded coordinates
    p = viz[i]['padded']; a = ax[3][c]
    a.plot(p[nzobj // 2, :], lw=0.9, label='horizontal, through z mid'); a.plot(p[:, nobj // 2], lw=0.9, label='vertical, through x mid')
    a.axhline(bg, color='k', ls=':', lw=0.8, label=f'pad level {bg:.3f}')
    for v in (margin + crop, nobj - margin - crop):
        a.axvline(v, color='r', ls='--', lw=0.6)
    a.set_title('profiles through the padded division', fontsize=9); a.set_xlabel('px'); a.legend(fontsize=7); a.grid(alpha=0.3)
plt.suptitle(f'{TAG}\nred = kept data (detector minus EDGE_CROP), yellow = BG_BOX (pad level), lime = PAGANIN_BG_BOX (global_bg, measured on the Paganin phase)', fontsize=10)
plt.tight_layout(); plt.show()
print(f'global_bg {global_bg:.4f} is the median of the lime box in row 2 at angle {viz[ks[0]]["theta"]:.1f} deg;  '
      f'after subtraction it is {np.median(rec_paganin[0, pbox[0]:pbox[1], pbox[2]:pbox[3]]):.2e}')


## Orthoslices of the initialised object `u`

Three cuts through the volume, real and imaginary. The **coronal** and
**sagittal** rows are the ones that expose padding problems — the blob lives
in the z-margin and is invisible in an axial slice. Red dashed lines mark the
margin boundary; anything bright outside them is padding, not sample.

In [ ]:
def _clim(a, lo=1, hi=99):
    v0, v1 = np.percentile(a, [lo, hi])
    return (float(v0), float(v1)) if v1 > v0 else (float(a.min()), float(a.max()) + 1e-9)

zc, yc, xc = nzobj // 2, nobj // 2, nobj // 2
views = [('axial  (x,y) @ z=%d' % zc,     lambda v: v[zc],        'xy'),
         ('coronal (x,z) @ y=%d' % yc,    lambda v: v[:, yc, :],  'xz'),
         ('sagittal (y,z) @ x=%d' % xc,   lambda v: v[:, :, xc],  'yz')]

fig, ax = plt.subplots(2, 3, figsize=(16, 10))
for r, (part, lbl) in enumerate(((obj_fbp.real, 'Re u'), (obj_fbp.imag, 'Im u'))):
    for c, (title, fn, kind) in enumerate(views):
        img = fn(part)
        a = ax[r][c]
        vlo, vhi = _clim(img)
        im = a.imshow(img, cmap='gray', vmin=vlo, vmax=vhi)
        a.set_title(f'{lbl} — {title}', fontsize=10)
        plt.colorbar(im, ax=a, fraction=0.046)
        if margin:
            # horizontal axis is x for xy/xz, y for yz; vertical is y or z
            vmar = marginz if kind in ('xz', 'yz') else margin
            nv   = nz if kind in ('xz', 'yz') else n
            for yln in (vmar, vmar + nv):
                a.axhline(yln, color='r', ls='--', lw=0.6)
            for xln in (margin, margin + n):
                a.axvline(xln, color='r', ls='--', lw=0.6)
        a.axis('off')
plt.suptitle(TAG, fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Quantitative version of the picture above: is the padding louder than the
# sample? A clean run has ratio < 1 and the hottest row inside the data band.
vert = obj_fbp[:, yc, :].real
if marginz:
    marg = np.abs(np.concatenate([vert[:marginz], vert[-marginz:]]))
    data = np.abs(vert[marginz:-marginz])
    zprof = np.abs(vert).max(axis=1)
    hot = int(zprof.argmax())
    print(f'z-margin  max {marg.max():9.3f}   mean {marg.mean():9.4f}')
    print(f'data band max {data.max():9.3f}   mean {data.mean():9.4f}')
    print(f'--> margin/data max ratio {marg.max() / (data.max() + 1e-12):.3f}'
          '   (>1 means the padding is louder than the sample)')
    print(f'--> hottest z row {hot} of {nzobj}; in the margin: '
          f'{bool(hot < marginz or hot >= nzobj - marginz)}')
else:
    print('NOBJ_FACTOR=1.0, no margin to check')

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(np.abs(vert).max(axis=1), lw=0.9)
if marginz:
    for v in (marginz, nzobj - marginz):
        ax[0].axvline(v, color='r', ls='--', lw=0.8)
ax[0].set_title('max |Re u| per z row  (spikes in the margin = padding artifact)', fontsize=10)
ax[0].set_xlabel('z'); ax[0].grid(alpha=0.3)

ax[1].plot(obj_fbp[zc, yc, :].real, lw=0.9, label='Re')
ax[1].plot(obj_fbp[zc, yc, :].imag, lw=0.9, label='Im')
if margin:
    for v in (margin, nobj - margin):
        ax[1].axvline(v, color='r', ls='--', lw=0.8)
ax[1].set_title(f'line profile through z={zc}, y={yc}', fontsize=10)
ax[1].set_xlabel('x'); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
plt.suptitle(TAG, fontsize=11)
plt.tight_layout()
plt.show()

## Diagnostic: why the FBP is real-first

`Tomo.RT` is a NUFFT adjoint and is not exactly real-preserving. On a complex sinogram it puts a small spurious imaginary part of the real channel into Im, where the true signal (`Re / DELTA_BETA`) is much weaker: the fine vertical lines. On a float32 sinogram it keeps only `.real`.

Measured here (BIN=3, DELTA_BETA=30): the FBP of a real sinogram on the complex path has imag/real = 3.7e-3 (the filter step alone: 2.6e-6), and Im from the complex sinogram is 11% off the ideal Re/30.

In [ ]:
# One 8-slice chunk of the REAL phase sinogram, three ways
z0 = nzobj // 2 - 4
R  = cp.asarray(rec_paganin[:, z0:z0 + 8, :])
rms = lambda x: float(cp.sqrt(cp.mean(cp.abs(x) ** 2)))

a = cl.fbp(R, 'ramp')                                                   # float32 in: RT keeps .real only
b = cl.fbp(R.astype('complex64'), 'ramp')                               # same data on the complex path, imag = 0
c = cl.fbp((R * (1 + 1j / DELTA_BETA)).astype('complex64'), 'ramp')     # complex sinogram R + iR/DELTA_BETA

print(f'FBP of a REAL sinogram on the complex path: rms(imag)/rms(real) = {rms(b.imag) / rms(a):.2e}   (0 if realness is preserved)')
print(f'Im of the complex-sinogram FBP vs the ideal Re/{DELTA_BETA:g}: off by {rms(c.imag - a / DELTA_BETA) / rms(a / DELTA_BETA):.1%}')

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
im = ax[0].imshow(b.imag[4].get(), cmap='gray'); plt.colorbar(im, ax=ax[0], fraction=0.046); ax[0].axis('off')
ax[0].set_title('imag part of the FBP of a REAL sinogram (complex path)', fontsize=10)
ax[1].plot((a / DELTA_BETA)[4, nobj // 2].get(), lw=0.9, label=f'ideal Re/{DELTA_BETA:g}')
ax[1].plot(c.imag[4, nobj // 2].get(), lw=0.9, label='Im from the complex sinogram')
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3); ax[1].set_title('x-profile through y=centre', fontsize=10)
plt.tight_layout(); plt.show()
del R, a, b, c
cp.get_default_memory_pool().free_all_blocks()


## Analysis (not part of the pipeline): is the slow background the same at every angle?

Nothing above is changed by the cells below; they only read the notebook's variables (`flat_field`, `sigma`, `bin2`, `nb`, `crop`, ...). For ~12 angles spread over the scan, a smooth surface (quadratic polynomial) is fitted to the **vacuum pixels** of the division, with outliers (the sample) rejected iteratively. Then the surfaces are compared between angles: if the shape is the same at every angle, it can be estimated once and divided out of every projection.

In [ ]:
def slow_background(n_angles=12, order=2):
    """Fit a smooth surface (polynomial of the given order) to the vacuum pixels of the division, at several angles.
    Reads the notebook's flat_field, sigma, bin2, nb, nz, n, crop and theta_ids; changes none of them."""
    ids = np.unique(np.linspace(0, ntheta - 1, n_angles).astype(int))          # indices into the angle subset
    yy, xx = np.mgrid[:nz, :n]
    X, Y = (xx - n / 2 + .5) / (n / 2), (yy - nz / 2 + .5) / (nz / 2)          # frame coordinates in [-1, 1]
    names = [f'x^{i} y^{j}' for i in range(order + 1) for j in range(order + 1 - i)]
    Phi = np.stack([X**i * Y**j for i in range(order + 1) for j in range(order + 1 - i)], -1).reshape(-1, len(names))
    inner = np.zeros((nz, n), bool); inner[crop:nz - crop, crop:n - crop] = True        # leave the stripped rim out
    out = {k: [] for k in ('ratio', 'mask', 'coef', 'surf')}
    with h5py.File(PATH_H5, 'r') as f:
        for i in ids:
            k = theta_ids[i]
            num = ndimage.gaussian_filter(cp.asarray(f['/exchange/pdata'][k].astype('float32')), sigma)
            den = ndimage.gaussian_filter(flat_field(k), sigma)
            r = bin2(num / (den + 1e-5), nb).get()                              # the division, as in the loop
            mask = inner.copy()
            for _ in range(8):                                                  # fit, drop outliers (the sample), refit
                cf = np.linalg.lstsq(Phi[mask.ravel()], r.ravel()[mask.ravel()], rcond=None)[0]
                res = r.ravel() - Phi @ cf
                s = 1.4826 * np.median(np.abs(res[mask.ravel()] - np.median(res[mask.ravel()])))
                mask = inner & (np.abs(res.reshape(nz, n)) < 2.5 * s)
            for key, val in zip(('ratio', 'mask', 'coef', 'surf'), (r, mask, cf, (Phi @ cf).reshape(nz, n))):
                out[key].append(val)
    out = {k: np.stack(v) for k, v in out.items()}
    out['ids'], out['names'] = ids, names
    out['level'] = np.array([np.median(r[m_]) for r, m_ in zip(out['ratio'], out['mask'])])    # vacuum level per angle
    return out


In [ ]:
sb = slow_background()
sb_k, sb_mean = len(sb['ids']), sb['surf'].mean(0)
sb_ptp = np.ptp(sb_mean)
sb_dev = np.sqrt(((sb['surf'] - sb_mean) ** 2).mean((1, 2)))
sb_shape = np.corrcoef((sb['surf'] - sb['surf'].mean((1, 2), keepdims=True)).reshape(sb_k, -1))
sb_off = np.triu_indices(sb_k, 1)
sb_lev = sb['level']; sb_ang = theta_deg[sb['ids']]

print(f"{sb_k} angles, kept {100 * sb['mask'].mean():.0f}% of the frame as vacuum")
print('fitted surface, coefficient per angle     ' + ''.join(f'{t:>9s}' for t in sb['names']))
print('  mean                                    ' + ''.join(f'{v:9.4f}' for v in sb['coef'].mean(0)))
print('  std over angles                         ' + ''.join(f'{v:9.4f}' for v in sb['coef'].std(0)))
print(f'peak-to-peak of the angle-averaged surface over the frame: {sb_ptp:.4f}')
print(f'angle-to-angle deviation of the surface (rms): mean {100 * sb_dev.mean() / sb_ptp:.0f}% of that peak-to-peak, max {100 * sb_dev.max() / sb_ptp:.0f}% (angle {sb_ang[sb_dev.argmax()]:.1f} deg)')
print(f'correlation of the SHAPE between angles: min {sb_shape[sb_off].min():.3f}, mean {sb_shape[sb_off].mean():.3f}')
print(f'vacuum LEVEL per angle: std {sb_lev.std():.4f}   (angle idx 0: {sb_lev[0]:.4f}, the others: {sb_lev[1:].mean():.4f} +- {sb_lev[1:].std():.4f})')

sb_fig, sb_ax = plt.subplots(2, 3, figsize=(17, 9))
sb_a = sb_ax[0][0]; sb_im = sb_a.imshow(sb['ratio'][0], cmap='gray', vmin=np.percentile(sb['ratio'][0], 1), vmax=np.percentile(sb['ratio'][0], 99))
sb_a.contour(sb['mask'][0], [0.5], colors='lime', linewidths=0.8); plt.colorbar(sb_im, ax=sb_a, fraction=0.046); sb_a.axis('off')
sb_a.set_title(f"division at angle {sb_ang[0]:.1f} deg; green = vacuum pixels used in the fit", fontsize=9)
sb_a = sb_ax[0][1]; sb_im = sb_a.imshow(sb_mean, cmap='viridis'); plt.colorbar(sb_im, ax=sb_a, fraction=0.046); sb_a.axis('off')
sb_a.set_title(f'angle-averaged fitted surface (peak-to-peak {sb_ptp:.3f})', fontsize=9)
sb_a = sb_ax[0][2]
for sb_s in sb['surf']:
    sb_a.plot(np.linspace(-1, 1, n), sb_s[nz // 2], color='C0', lw=0.8); sb_a.plot(np.linspace(-1, 1, nz), sb_s[:, n // 2], color='C1', lw=0.8, ls='--')
sb_a.set_title('fitted surface through the centre, all angles (blue: along x, orange dashed: along y)', fontsize=9); sb_a.grid(alpha=0.3)
sb_a = sb_ax[1][0]; sb_a.plot(sb_ang, sb_lev, 'o-', lw=0.8); sb_a.plot(sb_ang[0], sb_lev[0], 'ro')
sb_a.axhline(np.median(sb_lev), color='k', ls=':', lw=0.8); sb_a.set_xlabel('angle (deg)'); sb_a.grid(alpha=0.3)
sb_a.set_title('vacuum level per angle (red = angle idx 0, where pad level and global_bg are measured)', fontsize=9)
sb_a = sb_ax[1][1]; sb_a.bar(range(sb_k), 100 * sb_dev / sb_ptp); sb_a.set_xticks(range(sb_k)); sb_a.set_xticklabels([f'{v:.0f}' for v in sb_ang], fontsize=7)
sb_a.set_xlabel('angle (deg)'); sb_a.set_title('deviation of each angle\'s surface from the average, % of its peak-to-peak', fontsize=9); sb_a.grid(alpha=0.3)
sb_a = sb_ax[1][2]; sb_im = sb_a.imshow(sb_shape, vmin=sb_shape[sb_off].min(), vmax=1, cmap='magma'); plt.colorbar(sb_im, ax=sb_a, fraction=0.046)
sb_a.set_title('correlation of the surface shape between angles', fontsize=9)
plt.suptitle(TAG, fontsize=10); plt.tight_layout(); plt.show()


## Analysis on the Paganin phase (not part of the pipeline): what is angle-independent, and what does it do to the reconstruction?

Reads `rec_paganin` (the phase that goes into the FBP, after the `global_bg` subtraction) and `cl`; changes nothing above.

**Vacuum pixels, defined by rotation.** The sample rotates and the background does not, so a pixel the sample never covers has the same phase at every angle. The map `v` is the spread of each pixel's phase over all angles; "vacuum" = pixels with small `v`. No fit is involved in choosing them, and the mask is drawn on the map.

Then, in rad: (1) two estimates of the angle-independent background are made: **(a)** a smooth surface fitted to the vacuum pixels, **(b)** the measured per-pixel value on the vacuum pixels (this keeps the margin and the seam, which a smooth fit cannot) with the smooth surface used only under the sample; (2) the per-angle level is the median of `phase - background` over the vacuum pixels; (3) one 8-slice slab is reconstructed four ways with the same `cl`: the original sinogram, the measured background alone, (a) removed, and (b) removed. If the background is what makes the dark bowl, the background-alone reconstruction shows it and the corrected ones do not; if only (b) flattens it, the cause is structure that is not smooth.

In [ ]:
def paganin_background(order=2, vac_factor=3.0):
    """Angle-independent background of the Paganin sinogram and its effect on the FBP of one 8-slice slab.
    Reads rec_paganin and cl; changes neither."""
    nth, nzp, nxp = rec_paganin.shape
    m = np.median(rec_paganin.reshape(nth, -1), axis=1)                 # each angle's own level (median of the whole frame, mostly vacuum)
    flat = rec_paganin - m[:, None, None]                               # ... removed, so the level scatter does not hide the sample-free pixels
    v = flat.std(0)                                                     # spread of each pixel's phase over angle
    vac = v < vac_factor * np.percentile(v, 10)                         # the sample never covers these: same at every angle
    yy, xx = np.mgrid[:nzp, :nxp]
    X, Y = (xx - nxp / 2 + .5) / (nxp / 2), (yy - nzp / 2 + .5) / (nzp / 2)
    Phi = np.stack([X**i * Y**j for i in range(order + 1) for j in range(order + 1 - i)], -1).reshape(nzp * nxp, -1)
    b_data = np.median(flat, axis=0)                                    # per-pixel median over angles (meaningful only where vac)
    coef = np.linalg.lstsq(Phi[vac.ravel()], b_data.ravel()[vac.ravel()], rcond=None)[0]
    B = (Phi @ coef).reshape(nzp, nxp).astype('float32')                # (a) smooth surface: also extends under the sample
    Bh = np.where(vac, b_data, B).astype('float32')                     # (b) measured where vacuum (margin, seam, ...), smooth under the sample
    lev = lambda b: np.array([np.median(rec_paganin[k][vac] - b[vac]) for k in range(nth)])      # per-angle offset after removing b, rad
    level, levelh = lev(B), lev(Bh)
    sl = slice(nzp // 2 - 4, nzp // 2 + 4)
    recon = lambda s: cl.fbp(cp.asarray(np.ascontiguousarray(s, dtype='float32')), 'ramp').get()
    slab = rec_paganin[:, sl, :]
    return dict(v=v, vac=vac, B=B, Bh=Bh, level=level, levelh=levelh, fit_rms=float(np.sqrt(np.mean((b_data - B)[vac] ** 2))),
                fbp_orig=recon(slab),
                fbp_bg=recon(np.broadcast_to(Bh[sl], (nth, 8, nxp))),                                  # the measured background alone
                fbp_fix=recon(slab - B[sl][None] - level[:, None, None]),                              # (a) smooth removed
                fbp_fixh=recon(slab - Bh[sl][None] - levelh[:, None, None]))                           # (b) measured+smooth removed


In [ ]:
pb = paganin_background()
pb_ang = theta_deg
print(f"vacuum pixels (phase constant over angle): {100 * pb['vac'].mean():.0f}% of the padded frame;  v floor (10th percentile) {np.percentile(pb['v'], 10):.3f} rad")
print(f"smooth background (a): peak-to-peak {np.ptp(pb['B']):.3f} rad;  its residual against the measured one on the vacuum pixels {pb['fit_rms']:.3f} rad rms")
print(f"per-angle level: std {pb['levelh'].std():.3f} rad;  angle idx 0 is {pb['levelh'][0] - pb['levelh'][1:].mean():+.3f} rad from the mean of the others (their scatter {pb['levelh'][1:].std():.3f})")

pb_yy, pb_xx = np.mgrid[:nobj, :nobj] - (nobj - 1) / 2.0
pb_rr = np.hypot(pb_yy, pb_xx) / (nobj / 2)
pb_edges = np.linspace(0, 1.1, 56); pb_rc = 0.5 * (pb_edges[1:] + pb_edges[:-1])
pb_radial = lambda img: np.array([np.median(img[(pb_rr >= lo) & (pb_rr < hi)]) for lo, hi in zip(pb_edges[:-1], pb_edges[1:])])
pb_keys = ('fbp_orig', 'fbp_bg', 'fbp_fix', 'fbp_fixh')
pb_prof = {k: pb_radial(pb[k][4].real) for k in pb_keys}                  # slice 4 of the 8-slice slab = the middle
print('\nradial profile of the reconstruction (azimuthal median), peak-to-peak       r < 0.75   r < 1.0')
for pb_k, pb_lab in zip(pb_keys, ('original', 'measured background alone', '(a) smooth removed', '(b) measured + smooth removed')):
    print(f'  {pb_lab:32s}{np.ptp(pb_prof[pb_k][pb_rc < 0.75]):10.3f}{np.ptp(pb_prof[pb_k][pb_rc < 1.0]):10.3f}')
for pb_key, pb_lab in (('fbp_fix', '(a)'), ('fbp_fixh', '(b)')):
    print(f"  -> {pb_lab} removes {100 * (1 - np.ptp(pb_prof[pb_key][pb_rc < 0.75]) / np.ptp(pb_prof['fbp_orig'][pb_rc < 0.75])):.0f}% of the r < 0.75 variation")

pb_fig, pb_ax = plt.subplots(2, 3, figsize=(17, 9))
pb_a = pb_ax[0][0]; pb_im = pb_a.imshow(pb['v'], cmap='magma'); plt.colorbar(pb_im, ax=pb_a, fraction=0.046); pb_a.axis('off')
pb_a.contour(pb['vac'], [0.5], colors='cyan', linewidths=0.8)
pb_a.set_title('v = spread of each pixel\'s phase over angle (rad); cyan = outline of the vacuum pixels', fontsize=8)
pb_a = pb_ax[0][1]; pb_im = pb_a.imshow(pb['Bh'], cmap='viridis'); plt.colorbar(pb_im, ax=pb_a, fraction=0.046); pb_a.axis('off')
pb_a.set_title(f"(b) background: measured where vacuum, smooth under the sample (rad); peak-to-peak {np.ptp(pb['Bh']):.2f}", fontsize=8)
pb_a = pb_ax[0][2]; pb_a.plot(pb_ang, pb['levelh'], 'o-', lw=0.6, ms=2); pb_a.plot(pb_ang[0], pb['levelh'][0], 'ro')
pb_a.set_xlabel('angle (deg)'); pb_a.set_title('per-angle level of the phase after (b) (rad); red = angle idx 0', fontsize=9); pb_a.grid(alpha=0.3)
pb_lim = np.percentile(pb['fbp_orig'][4].real, [1, 99])
for pb_a, pb_key, pb_name in ((pb_ax[1][0], 'fbp_orig', 'reconstruction: original'), (pb_ax[1][1], 'fbp_fixh', 'reconstruction: (b) removed')):
    pb_im = pb_a.imshow(pb[pb_key][4].real, cmap='gray', vmin=pb_lim[0], vmax=pb_lim[1]); plt.colorbar(pb_im, ax=pb_a, fraction=0.046); pb_a.axis('off'); pb_a.set_title(pb_name + ' (same grey scale)', fontsize=9)
pb_a = pb_ax[1][2]
for pb_key, pb_col, pb_lab in (('fbp_orig', 'k', 'original'), ('fbp_bg', 'r', 'measured background alone'), ('fbp_fix', 'g', '(a) smooth removed'), ('fbp_fixh', 'b', '(b) measured + smooth removed')):
    pb_a.plot(pb_rc, pb_prof[pb_key], color=pb_col, label=pb_lab)
pb_a.axvline(n / nobj, color='gray', ls='--', lw=0.8); pb_a.axvline(np.sqrt(MASK_R), color='gray', ls=':', lw=0.8)
pb_a.set_xlabel('normalised radius (dashed: detector edge, dotted: FBP mask)'); pb_a.legend(fontsize=8); pb_a.grid(alpha=0.3)
pb_a.set_title('radial profile of the reconstruction (azimuthal median)', fontsize=9)
plt.suptitle(TAG, fontsize=10); plt.tight_layout(); plt.show()


## Analysis (not part of the pipeline): SVD of a handful of frames

Takes ~16 angles spread over the scan and stacks the **log of the division** (detector frame, before the alignment shift, rim stripped) into a matrix `M` (frames x pixels). Each frame's own mean (its level) is removed first. Then `M = U S V^T`.

The slow background is the same in every frame, so it should be the **first mode**: a large singular value, a spatial image `V[0]` that looks like a smooth tilt/curvature, and a coefficient `S[0] U[:, 0]` that is the same at every angle. The sample moves, so the later modes should have coefficients that change with angle.

Nothing above is changed; the cells only read `flat_field`, `sigma`, `bin2`, `nb`, `nz`, `n`, `crop`, `theta_ids`, `theta_deg`.

In [ ]:
def svd_stack(n_frames=64):
    """SVD of the stack of divisions (log, detector frame, rim stripped) at a handful of angles.
    Reads flat_field, sigma, bin2, nb, nz, n, crop and theta_ids; changes none of them."""
    ids = np.unique(np.linspace(0, ntheta - 1, n_frames).astype(int))          # indices into the angle subset
    rows = []
    with h5py.File(PATH_H5, 'r') as f:
        for i in ids:
            k = theta_ids[i]
            num = ndimage.gaussian_filter(cp.asarray(f['/exchange/pdata'][k].astype('float32')), sigma)
            den = ndimage.gaussian_filter(flat_field(k), sigma)
            r = bin2(num / (den + 1e-5), nb).get()[crop:nz - crop, crop:n - crop]    # the division as in the pipeline, rim stripped
            rows.append(np.log(r).ravel())
    M = np.stack(rows)                                                           # frames x pixels
    level = M.mean(1)                                                            # each frame's own level
    U, s, Vt = np.linalg.svd(M - level[:, None], full_matrices=False)            # level removed, NOT centred over frames
    sgn = np.sign(U.mean(0)); sgn[sgn == 0] = 1                                  # fix the arbitrary sign of each mode
    return dict(ids=ids, level=level, U=U * sgn, s=s, Vt=Vt * sgn[:, None], shape=(nz - 2 * crop, n - 2 * crop))


In [ ]:
sv = svd_stack()
sv_h, sv_w = sv['shape']; sv_ang = theta_deg[sv['ids']]; sv_k = len(sv['ids'])
sv_coef = sv['U'] * sv['s']                                                      # frames x modes: how strongly each mode is present in each frame
print(f"{sv_k} frames of {sv_h}x{sv_w} px;  per-frame level: std {sv['level'].std():.4f}, frame 0 is {sv['level'][0] - sv['level'][1:].mean():+.4f} from the others")
print('singular values:', ' '.join(f'{v:.2f}' for v in sv['s'][:8]), f"   (mode 1 / mode 2 = {sv['s'][0] / sv['s'][1]:.1f})")
print(f"mode 1 coefficient over angle: mean {sv_coef[:, 0].mean():.2f}, std/mean {100 * sv_coef[:, 0].std() / abs(sv_coef[:, 0].mean()):.0f}%   (small = common to every angle)")
print('modes 2-4 coefficient over angle: std/mean-of-|.|  ' + '  '.join(f"{100 * sv_coef[:, j].std() / np.abs(sv_coef[:, j]).mean():.0f}%" for j in (1, 2, 3)) + "   (large = changes with angle)")

sv_modes = [sv['Vt'][j].reshape(sv_h, sv_w) for j in range(4)]
sv_lp = ndimage.gaussian_filter(cp.asarray(sv_modes[0]), 0.05 * sv_w).get()      # mode 1, only its slow part
sv_fig, sv_ax = plt.subplots(2, 4, figsize=(18, 9))
sv_a = sv_ax[0][0]; sv_a.semilogy(sv['s'], 'o-'); sv_a.set_xlabel('mode'); sv_a.grid(alpha=0.3); sv_a.set_title('singular values (level removed)', fontsize=9)
sv_a = sv_ax[0][1]
for j in range(4): sv_a.plot(sv_ang, sv_coef[:, j], 'o-', ms=3, lw=0.8, label=f'mode {j + 1}')
sv_a.set_xlabel('angle (deg)'); sv_a.legend(fontsize=8); sv_a.grid(alpha=0.3); sv_a.set_title('coefficient of each mode in each frame', fontsize=9)
for sv_a, img, name, cm in ((sv_ax[0][2], sv_modes[0], 'mode 1 (spatial)', 'viridis'), (sv_ax[0][3], sv_lp, 'mode 1, slow part only', 'viridis'),
                            (sv_ax[1][0], sv_modes[1], 'mode 2', 'RdBu_r'), (sv_ax[1][1], sv_modes[2], 'mode 3', 'RdBu_r'),
                            (sv_ax[1][2], sv_modes[3], 'mode 4', 'RdBu_r'), (sv_ax[1][3], sv_modes[0] - sv_lp, 'mode 1 minus its slow part', 'RdBu_r')):
    sv_lim = np.abs(img).max() if cm == 'RdBu_r' else None
    sv_im = sv_a.imshow(img, cmap=cm, vmin=None if sv_lim is None else -sv_lim, vmax=sv_lim); plt.colorbar(sv_im, ax=sv_a, fraction=0.046); sv_a.axis('off'); sv_a.set_title(name, fontsize=9)
plt.suptitle(TAG, fontsize=10); plt.tight_layout(); plt.show()


## Diagnostic: is the high-frequency code residue a sub-pixel `shifts_code` error?

`/exchange/shifts_code` is integer-valued at every one of the 7200 angles (max fractional part
exactly `0.0`), because stage 3's shift finder takes a plain `cp.argmax` of the cross-correlation
with no sub-pixel refinement.

`code_window` above therefore never exercises its `fourier_shift_2d` branch -- the remainder
`shifts_code[k] - int(round(shifts_code[k]))` it passes in is identically zero. The modelled flat
field is placed on integer pixels only, so it sits up to +-0.5 px off the measurement, and the
division cannot cancel the code where it is steepest.

Below: scan a sub-pixel offset over the code window, measure the high-passed residue left in the
division, and see whether the minimum sits away from (0, 0) and what it buys. Nothing here feeds
`rec_paganin` -- the production loop above has already run.

In [ ]:
# Offset-aware twins of cell 3's code_window / flat_field. At (0, 0) they ARE cell 3's versions
# (asserted at the bottom); a non-zero offset is added to shifts_code[k] BEFORE the integer /
# remainder split, so the sub-pixel branch the integer shifts_code leaves dead actually runs.
def code_window_off(k, drow=0.0, dcol=0.0, G=8):
    s_r = float(shifts_code[k, 0]) + drow
    s_c = float(shifts_code[k, 1]) + dcol
    dy, dx = int(round(s_r)), int(round(s_c))
    cr, cc = ncode_full // 2 - dy, ncode_full // 2 - dx
    w = code_gpu[cr - nz_full // 2 - G: cr + nz_full // 2 + G, cc - n_full // 2 - G: cc + n_full // 2 + G]
    w = fourier_shift_2d(w, dc=s_c - dx, dr=s_r - dy)
    return rescale_about_centre(w, CODE_SCALE)[G:-G, G:-G]


def flat_field_off(k, drow=0.0, dcol=0.0):
    return cp.abs(cl_prop.D(cl_prop_code.D(prb_gpu * code_window_off(k, drow, dcol), 0), 0)) ** 2


SIGMA_HP = 2.0          # binned px. The smooth part removed before the residue is measured, so the
                        # metric sees the code texture and not the object's slow phase ramp.


def hp(a):
    return a - ndimage.gaussian_filter(a, SIGMA_HP)


def ratio_off(num_blur, k, drow=0.0, dcol=0.0):
    """The division exactly as the production loop does it, code window offset by (drow, dcol).
    num_blur is the already-blurred measurement, hoisted out so the scan reuses it."""
    den = ndimage.gaussian_filter(flat_field_off(k, drow, dcol), sigma)
    return bin2(num_blur / (den + 1e-5), nb)


assert float(cp.abs(code_window_off(theta_ids[0]) - code_window(theta_ids[0])).max()) == 0.0
print(f'offset twins == cell 3 at (0, 0);  blur sigma {sigma:.2f} unbinned px, '
      f'high-pass sigma {SIGMA_HP:g} binned px')

In [ ]:
# Scan the sub-pixel offset. Metric is the RMS of the high-passed division: with the code cancelled
# the ratio keeps only the object and noise, and a misregistration adds a code-shaped term on top.
SCAN_HALF, SCAN_N = 0.6, 7                  # +- px about the integer shift, grid points per axis
SCAN_K = VIZ_K[:3]                          # which of the viz angles to test

grid = np.linspace(-SCAN_HALF, SCAN_HALF, SCAN_N)
ctr  = SCAN_N // 2
assert grid[ctr] == 0.0, 'grid must straddle 0 so the centre is the production value'

scan = {}
with h5py.File(PATH_H5, 'r') as f:
    pdata = f['/exchange/pdata']
    for i in SCAN_K:
        k = theta_ids[i]
        num_blur = ndimage.gaussian_filter(cp.asarray(pdata[k].astype('float32')), sigma)
        surf = np.zeros((SCAN_N, SCAN_N), dtype='float32')
        for a, dr in enumerate(grid):
            for b, dc in enumerate(grid):
                surf[a, b] = float(cp.sqrt((hp(ratio_off(num_blur, k, dr, dc)) ** 2).mean()))
        a, b = np.unravel_index(surf.argmin(), surf.shape)
        scan[i] = dict(surf=surf, best=(float(grid[a]), float(grid[b])),
                       rms0=float(surf[ctr, ctr]), rms1=float(surf[a, b]))
        edge = (a in (0, SCAN_N - 1)) or (b in (0, SCAN_N - 1))
        print(f'  angle {i:4d} (theta {theta_deg[i]:7.2f} deg)  best ({grid[a]:+.2f}, {grid[b]:+.2f}) px'
              f'   hp-RMS {surf[ctr, ctr]:.5f} -> {surf[a, b]:.5f}'
              f'  ({100 * (surf[a, b] / surf[ctr, ctr] - 1):+.1f}%)'
              + ('   <-- ON THE GRID EDGE, widen SCAN_HALF' if edge else ''))
cp.get_default_memory_pool().free_all_blocks()

In [ ]:
# The picture behind the numbers, for the first scanned angle: the metric surface, then the division
# and its high-pass at the integer shift vs at the refined one. Both high-pass panels share one
# colour scale, so "less code texture" is a fair comparison and not a stretch artefact.
i = SCAN_K[0]
k = theta_ids[i]
dr, dc = scan[i]['best']
with h5py.File(PATH_H5, 'r') as f:
    num_blur = ndimage.gaussian_filter(cp.asarray(f['/exchange/pdata'][k].astype('float32')), sigma)
r_int, r_ref = ratio_off(num_blur, k, 0.0, 0.0), ratio_off(num_blur, k, dr, dc)
h_int, h_ref = hp(r_int).get(), hp(r_ref).get()
r_int, r_ref = r_int.get(), r_ref.get()
cp.get_default_memory_pool().free_all_blocks()

fig, ax = plt.subplots(1, 5, figsize=(26, 5.2))
im = ax[0].imshow(scan[i]['surf'], cmap='viridis', extent=[grid[0], grid[-1], grid[-1], grid[0]])
ax[0].plot(0, 0, 'wx', ms=11, label='integer (production)')
ax[0].plot(dc, dr, 'r+', ms=14, label=f'best ({dr:+.2f}, {dc:+.2f})')
ax[0].set(title='hp-RMS vs code offset', xlabel='dcol [px]', ylabel='drow [px]')
ax[0].legend(loc='upper right', fontsize=8)
fig.colorbar(im, ax=ax[0], fraction=0.046)

vlo, vhi = _clim(r_int)
for slot, img, ttl in ((1, r_int, 'ratio, integer shift'), (2, r_ref, f'ratio, {dr:+.2f}/{dc:+.2f} px')):
    im = ax[slot].imshow(img, cmap='gray', vmin=vlo, vmax=vhi)
    ax[slot].set_title(ttl)
    fig.colorbar(im, ax=ax[slot], fraction=0.046)

vlo, vhi = _clim(h_int)
for slot, img, ttl in ((3, h_int, f'high-pass, integer   RMS {scan[i]["rms0"]:.5f}'),
                       (4, h_ref, f'high-pass, refined   RMS {scan[i]["rms1"]:.5f}')):
    im = ax[slot].imshow(img, cmap='gray', vmin=vlo, vmax=vhi)
    ax[slot].set_title(ttl)
    fig.colorbar(im, ax=ax[slot], fraction=0.046)

fig.suptitle(f'angle {i} (theta {theta_deg[i]:.2f} deg), {TAG}', y=1.02)
plt.tight_layout()
plt.show()

## Diagnostic: unblurred division, does the residue look like a registration error?

Production blurs both the measurement and the modelled flat field (sigma ~14.4 unbinned px,
`smooth_fwhm=17`) before dividing. That blur also smooths over whatever a registration error
would leave behind, so it is the wrong place to look for one.

A pure sub-pixel misregistration has a specific first-order signature: if the modelled flat field
is evaluated at the wrong position by (dy, dx), `flat_wrong = flat_true + dy*dflat/dy + dx*dflat/dx`,
and since `frame ~= flat_true` where there is no sample, the division residue is approximately
`frame/flat_wrong - 1 ~= -(dy*dflat/dy + dx*dflat/dx) / flat_true`
i.e. proportional to the flat field's OWN spatial gradient, not to the object or to noise. Below:
divide with NO blur, then regress the residue against the flat field's measured gradient components
to read off an implied (dy, dx) directly from the data, independent of the earlier shift-scan cells.

In [ ]:
def ratio_raw(k):
    """Division with no smoothing on either side, at full (unbinned) resolution."""
    with h5py.File(PATH_H5, 'r') as f:
        frame = cp.asarray(f['/exchange/pdata'][k].astype('float32'))
    flat = flat_field(k)
    return frame / (flat + 1e-5), frame, flat


def flat_grad(flat):
    """Central-difference gradient of the flat field, same grid as ratio_raw."""
    gy, gx = cp.gradient(flat)
    return gy, gx


def implied_shift(k):
    """Least-squares fit of (ratio_raw - 1) against the flat field's own gradient components,
    restricted to the kept region (rim stripped, same EDGE_CROP as production) so the fit isn't
    dominated by the unreliable edge. Returns (dy, dx) in unbinned px -- the sign convention is
    `flat_wrong - flat_true ~= dy*dflat/dy + dx*dflat/dx`, so a positive fitted dy means the
    MODEL needs to move by +dy to match the measurement, same sense as shifts_code's own units.
    """
    r, frame, flat = ratio_raw(k)
    gy, gx = flat_grad(flat)
    resid = (r - 1) * flat                      # un-normalise: residue ~= dy*gy + dx*gx directly
    sl = (slice(crop, nz_full - crop), slice(crop, n_full - crop))
    A = cp.stack([gy[sl].ravel(), gx[sl].ravel()], axis=1)
    b = resid[sl].ravel()
    sol, *_ = cp.linalg.lstsq(A.astype('float64'), b.astype('float64'), rcond=None)
    dy, dx = float(sol[0]), float(sol[1])
    pred = (dy * gy + dx * gx)[sl]
    r2 = 1 - float(cp.sum((b - pred.ravel()) ** 2) / cp.sum((b - b.mean()) ** 2))
    return dy, dx, r2


print(f'{"angle":>6} {"theta":>8} | {"implied dy":>11} {"implied dx":>11} {"R^2":>7}')
implied = {}
for i in VIZ_K:
    k = theta_ids[i]
    dy, dx, r2 = implied_shift(k)
    implied[i] = (dy, dx, r2)
    print(f'{i:6d} {theta_deg[i]:8.2f} | {dy:11.4f} {dx:11.4f} {r2:7.3f}')
cp.get_default_memory_pool().free_all_blocks()

In [ ]:
# The picture for the first VIZ_K angle: unblurred ratio, its implied-shift fit, and the residual
# after subtracting that fit -- if the R^2 above is high, this panel should look mostly flat.
i = VIZ_K[0]
k = theta_ids[i]
dy, dx, r2 = implied[i]
r, frame, flat = ratio_raw(k)
gy, gx = flat_grad(flat)
resid = ((r - 1) * flat).get()
fit = (dy * gy + dx * gx).get()
sl = (slice(crop, nz_full - crop), slice(crop, n_full - crop))

fig, ax = plt.subplots(1, 4, figsize=(21, 5))
panels = [('ratio_raw (unblurred)', r.get()),
          ('residue = (ratio_raw-1)*flat', resid),
          (f'fit: dy={dy:+.3f} dx={dx:+.3f} px', fit),
          (f'residue - fit  (R^2={r2:.3f})', resid - fit)]
for a, (title, img) in zip(ax, panels):
    sub = img[sl]
    im = a.imshow(sub, cmap='gray', vmin=np.percentile(sub, 1), vmax=np.percentile(sub, 99))
    a.set_title(title, fontsize=10)
    fig.colorbar(im, ax=a, fraction=0.046)
fig.suptitle(f'angle {i} (theta {theta_deg[i]:.2f} deg), {TAG}', y=1.02)
plt.tight_layout()
plt.show()

## Attempt: match the model's contrast to the measurement before dividing

If the division leaves code-shaped residue, then `flat_field(k)` is not what the detector actually
recorded with no sample. Suppose the difference is linear and shift-invariant, i.e. the true flat
field is `H * den` for some unknown filter `H` (convolution). Then `H` is measurable from the data
without knowing the object, because the object and the code are uncorrelated: the code window moves
by hundreds to thousands of px between angles while the object rotates, so averaging the
cross-spectrum over angles leaves the code's contribution and averages the object's toward zero.

    H(f) = < A(f) conj(B(f)) > / < |B(f)|^2 >

with `A` the measurement and `B` the model, both reduced to their RELATIVE modulation about a smooth
envelope first (`x / blur(x) - 1`), so the illumination shape and the overall scale drop out and what
is left is the code's own contrast. This is the least-squares filter taking the model onto the
measurement -- the same estimator as a Wiener/system-identification transfer function.

The SHAPE of H says what is wrong, which is the point of doing it this way rather than assuming:

  * H flat and != 1        the code's modulation depth is wrong (code.npy amplitude/phase scale)
  * H decaying smoothly    an unmodelled blur on the measurement (detector PSF, finite source size)
  * H oscillating          distance_code is wrong -- Fresnel transfer goes as cos/sin(pi*lam*z*f^2),
                           so a wrong z swings contrast up and down with frequency
  * arg(H) a linear ramp   a residual registration offset between model and measurement

The band that matters is set by the code itself. A 1-2 um code feature magnified by
fdd/|z1_code| = 72.1 onto a 3.0375 um pixel is 24-47 detector px, i.e. f = 0.021-0.042 cyc/px, and
the production blur (sigma 7.22 px) still passes 64% down to 16% there, so residue in that band
survives into the Paganin input. H is estimated on UNBLURRED frames so the full band is visible, then
applied to `den` before the production blur.

In [ ]:
# --- estimate H(f) ------------------------------------------------------------------------------
N_TF      = 8        # angles averaged into the cross-spectrum
TF_CROP   = 128      # unbinned px trimmed off each side before the FFT (rim is unreliable)
ENV_SIGMA = 32.0     # unbinned px; the smooth envelope divided out of BOTH sides first

tf_ids = np.unique(np.linspace(0, ntheta - 1, N_TF).astype(int))
tf_nz, tf_n = nz_full - 2 * TF_CROP, n_full - 2 * TF_CROP
tf_win = cp.asarray(np.outer(np.hanning(tf_nz), np.hanning(tf_n)).astype('float32'))


def tf_modulation(x):
    """Relative modulation about a smooth envelope: kills the illumination shape and the scale,
    keeps the code's own contrast. Cropped and Hann-windowed so the FFT sees no frame edge."""
    env = ndimage.gaussian_filter(x, ENV_SIGMA)
    m = x / (env + 1e-6) - 1.0
    return m[TF_CROP:nz_full - TF_CROP, TF_CROP:n_full - TF_CROP] * tf_win


tf_cross = cp.zeros((tf_nz, tf_n), dtype='complex64')
tf_power = cp.zeros((tf_nz, tf_n), dtype='float32')
with h5py.File(PATH_H5, 'r') as f:
    for j, i in enumerate(tf_ids):
        k = theta_ids[i]
        A = cp.fft.fft2(tf_modulation(cp.asarray(f['/exchange/pdata'][k].astype('float32'))))
        B = cp.fft.fft2(tf_modulation(flat_field(k)))
        tf_cross += A * cp.conj(B)
        tf_power += (cp.abs(B) ** 2).astype('float32')
        print(f'  {j + 1}/{len(tf_ids)}  angle {i:5d} (theta {theta_deg[i]:7.2f} deg)')
cp.get_default_memory_pool().free_all_blocks()

# radial average of the cross-spectrum and the power SEPARATELY, then divide (never average a ratio)
tf_fy = cp.fft.fftfreq(tf_nz).astype('float32')
tf_fx = cp.fft.fftfreq(tf_n).astype('float32')
tf_fr = cp.sqrt(cp.meshgrid(tf_fy, tf_fx, indexing='ij')[0] ** 2
                + cp.meshgrid(tf_fy, tf_fx, indexing='ij')[1] ** 2)
TF_NBIN = 160
tf_edges = cp.linspace(0, 0.5, TF_NBIN + 1)
tf_idx = cp.clip(cp.digitize(tf_fr.ravel(), tf_edges) - 1, 0, TF_NBIN - 1)
tf_cre = cp.bincount(tf_idx, tf_cross.real.ravel(), minlength=TF_NBIN)
tf_cim = cp.bincount(tf_idx, tf_cross.imag.ravel(), minlength=TF_NBIN)
tf_pw  = cp.bincount(tf_idx, tf_power.ravel(),      minlength=TF_NBIN)
tf_cnt = cp.bincount(tf_idx, minlength=TF_NBIN)

tf_H = ((tf_cre + 1j * tf_cim) / (tf_pw + 1e-20)).get()
tf_f = (0.5 * (tf_edges[1:] + tf_edges[:-1])).get()
tf_pw_np, tf_cnt_np = tf_pw.get(), tf_cnt.get()
tf_band = (tf_f >= 0.021) & (tf_f <= 0.042)          # the code's own band

print()
print(f'{"f_px":>8} {"period":>8} {"|H|":>8} {"arg(H) deg":>11} {"model power (rel)":>18}')
for ff in (0.005, 0.010, 0.021, 0.030, 0.042, 0.060, 0.10, 0.20, 0.35):
    b = int(np.argmin(np.abs(tf_f - ff)))
    print(f'{tf_f[b]:8.4f} {1 / tf_f[b]:8.1f} {abs(tf_H[b]):8.4f} {np.degrees(np.angle(tf_H[b])):11.2f}'
          f' {tf_pw_np[b] / tf_pw_np.max():18.3e}')
print()
print(f'|H| averaged over the code band 0.021-0.042 cyc/px = {np.abs(tf_H[tf_band]).mean():.4f}')
print('  (1.0 = model already matches the measurement there; <1 = model over-predicts the contrast)')

In [ ]:
# --- what does the shape say? -------------------------------------------------------------------
fig, ax = plt.subplots(1, 3, figsize=(19, 5))

a = ax[0]
a.plot(tf_f, np.abs(tf_H), 'k-', lw=1.2, label='|H| measured')
a.axhline(1.0, color='g', ls='--', lw=0.9, label='1.0 = model correct')
a.axvspan(0.021, 0.042, color='orange', alpha=0.18, label='code band (24-47 px)')
a.set_xlim(0, 0.25); a.set_ylim(0, max(1.6, float(np.nanmax(np.abs(tf_H[tf_f < 0.25]))) * 1.1))
a.set_xlabel('f [cyc/px]'); a.set_ylabel('|H|'); a.legend(fontsize=8); a.grid(alpha=0.3)
a.set_title('transfer function magnitude', fontsize=10)

a = ax[1]
a.plot(tf_f, np.degrees(np.angle(tf_H)), 'k-', lw=1.0)
a.axhline(0, color='g', ls='--', lw=0.9)
a.axvspan(0.021, 0.042, color='orange', alpha=0.18)
a.set_xlim(0, 0.25); a.set_xlabel('f [cyc/px]'); a.set_ylabel('arg(H) [deg]'); a.grid(alpha=0.3)
a.set_title('phase -- a linear ramp here means a residual registration offset', fontsize=10)

a = ax[2]
a.semilogy(tf_f, tf_pw_np / tf_pw_np.max(), 'b-', lw=1.0, label="model's own power |B|^2")
a.axvspan(0.021, 0.042, color='orange', alpha=0.18)
a.set_xlim(0, 0.25); a.set_xlabel('f [cyc/px]'); a.legend(fontsize=8); a.grid(alpha=0.3)
a.set_title('where the code actually has energy\n(H is only meaningful where this is not tiny)', fontsize=10)

plt.suptitle(TAG, fontsize=10); plt.tight_layout(); plt.show()

In [ ]:
# --- apply H to the model on a HELD-OUT angle and see whether the residue drops -----------------
TF_HP_SIGMA = 24.0        # unbinned px; high-pass that isolates the 24-60 px band the code lives in

i_tf_test = int(tf_ids[len(tf_ids) // 2]) + max(1, ntheta // (4 * len(tf_ids)))
i_tf_test = min(i_tf_test, ntheta - 1)
assert i_tf_test not in set(tf_ids.tolist()), 'landed on a training angle; nudge the offset'
k_tf = theta_ids[i_tf_test]

# radial |H| lifted back onto the 2D grid. Magnitude only: a phase correction would be a shift,
# which is the registration question, not the contrast question this cell is about.
tf_Hmag = cp.asarray(np.interp(cp.asnumpy(tf_fr).ravel(), tf_f, np.abs(tf_H)
                               ).reshape(tf_nz, tf_n).astype('float32'))

with h5py.File(PATH_H5, 'r') as f:
    a_full = cp.asarray(f['/exchange/pdata'][k_tf].astype('float32'))
b_full = flat_field(k_tf)

# rebuild the model with its modulation rescaled by H, at full frame size
b_env = ndimage.gaussian_filter(b_full, ENV_SIGMA)
b_mod = b_full / (b_env + 1e-6) - 1.0
b_sub = b_mod[TF_CROP:nz_full - TF_CROP, TF_CROP:n_full - TF_CROP]
b_mod_c = cp.real(cp.fft.ifft2(tf_Hmag * cp.fft.fft2(b_sub)))
b_corr = b_full.copy()
b_corr[TF_CROP:nz_full - TF_CROP, TF_CROP:n_full - TF_CROP] = (
    b_env[TF_CROP:nz_full - TF_CROP, TF_CROP:n_full - TF_CROP] * (1.0 + b_mod_c))

num_b = ndimage.gaussian_filter(a_full, sigma)
r_before = num_b / (ndimage.gaussian_filter(b_full, sigma) + 1e-5)
r_after  = num_b / (ndimage.gaussian_filter(b_corr, sigma) + 1e-5)

def tf_hp(x):
    return x - ndimage.gaussian_filter(x, TF_HP_SIGMA)

sl = (slice(TF_CROP, nz_full - TF_CROP), slice(TF_CROP, n_full - TF_CROP))
rms0 = float(cp.sqrt((tf_hp(r_before)[sl] ** 2).mean()))
rms1 = float(cp.sqrt((tf_hp(r_after)[sl] ** 2).mean()))
print(f'held-out angle {i_tf_test} (theta {theta_deg[i_tf_test]:.2f} deg)')
print(f'high-pass RMS of the division (sigma {TF_HP_SIGMA:g} px): '
      f'{rms0:.5f} -> {rms1:.5f}   ({100 * (rms1 / rms0 - 1):+.1f}%)')

r0, r1 = r_before[sl].get(), r_after[sl].get()
h0, h1 = tf_hp(r_before)[sl].get(), tf_hp(r_after)[sl].get()
cp.get_default_memory_pool().free_all_blocks()

fig, ax = plt.subplots(1, 4, figsize=(22, 5.4))
vlo, vhi = np.percentile(r0, [1, 99])
for a_i, img, t in ((ax[0], r0, 'division, model as-is'), (ax[1], r1, 'division, model x H')):
    im = a_i.imshow(img, cmap='gray', vmin=vlo, vmax=vhi); a_i.set_title(t, fontsize=10)
    fig.colorbar(im, ax=a_i, fraction=0.046)
hlo, hhi = np.percentile(h0, [1, 99])
for a_i, img, t in ((ax[2], h0, f'high-pass, as-is   RMS {rms0:.5f}'),
                    (ax[3], h1, f'high-pass, x H     RMS {rms1:.5f}')):
    im = a_i.imshow(img, cmap='gray', vmin=hlo, vmax=hhi); a_i.set_title(t, fontsize=10)
    fig.colorbar(im, ax=a_i, fraction=0.046)
plt.suptitle(f'angle {i_tf_test}, {TAG}', y=1.02); plt.tight_layout(); plt.show()

## Attempt: remove a PER-ANGLE low-order surface, fitted on the CROPPED division, then pad

Reworked order. The earlier version fitted the surface to the Paganin phase over the padded frame
and subtracted it there, so the pad ramp (whose end level was measured once, at angle 0) never saw
the correction and stopped matching the interior at every other angle.

Now, per angle, inside the production loop:

1. the division `ratio` is cropped to the kept rectangle (EDGE_CROP stripped), exactly as before;
2. an order-`LF_ORDER` polynomial is fitted to `log(kept)` with iterative sigma clipping (the sample
   is rejected), using only that rectangle, no synthetic pad pixels;
3. `kept / exp(S)` is the corrected division, whose vacuum sits at 1 at EVERY angle (this also
   removes the per-angle level scatter, which the old code left in);
4. only then it is padded, and the pad ramp ends at `bg = 1.0`, so there is no angle-0 measurement
   any more;
5. reflect pad, Paganin and the alignment shift as in the production loop.

Why divide the ratio rather than subtract on the phase: `multiPaganin` takes `log()` after its
filter, so for a smooth surface `log(ratio / S) = log(ratio) - log(S)`; correcting before the
filter makes the surface part of what Paganin sees, instead of something added afterwards.

Deliberate differences from the production loop (flagged, not hidden): the pad level is 1.0 (not
BG_BOX of angle 0) and `global_bg` is not subtracted, because both are now 1 / 0 by construction.
The printed lime-box median of angle 0 shows how close to 0 the result is without it.

`LF_ORDER = 1` (plane only) is the conservative fallback if order 2 starts eating the sample;
the kept fraction printed below is the number to watch.

In [ ]:
# --- per-angle surface on the CROPPED division, then pad, then Paganin ----------------------------
LF_ORDER = 4       # 1 = plane (tilt only), 2 = + quadratic (bowl)
LF_NITER = 8       # sigma-clipping iterations
LF_KAPPA = 2.5     # clip threshold in robust sigma

kh, kw = nz - 2 * crop, n - 2 * crop                              # the kept (cropped) division
_yy, _xx = np.mgrid[:kh, :kw]
_X = (_xx - kw / 2 + .5) / (kw / 2)
_Y = (_yy - kh / 2 + .5) / (kh / 2)
lf_names = [f'x^{i} y^{j}' for i in range(LF_ORDER + 1) for j in range(LF_ORDER + 1 - i)]
lf_Phi = np.stack([(_X ** i * _Y ** j).ravel() for i in range(LF_ORDER + 1) for j in range(LF_ORDER + 1 - i)], axis=1)


def lf_fit(logr):
    """Robust low-order fit to log(kept division). Returns (surface, coefs, kept_mask), all on the cropped grid."""
    v = logr.ravel()
    m = np.ones(v.size, bool)
    cf = None
    for _ in range(LF_NITER):
        cf = np.linalg.lstsq(lf_Phi[m], v[m], rcond=None)[0]
        res = v - lf_Phi @ cf
        s = 1.4826 * np.median(np.abs(res[m] - np.median(res[m])))
        if not np.isfinite(s) or s == 0:
            break
        m = np.abs(res) < LF_KAPPA * s
    return (lf_Phi @ cf).reshape(kh, kw), cf, m.reshape(kh, kw)


rec_lf = np.zeros((ntheta, nzobj, nobj), dtype='float32')
lf_coefs = np.zeros((ntheta, len(lf_names)), dtype='float32')
lf_kept = np.zeros(ntheta, dtype='float32')
lf_ptp = np.zeros(ntheta, dtype='float32')
viz_lf = {}
with h5py.File(PATH_H5, 'r') as f:
    pdata = f['/exchange/pdata']
    for i, k in enumerate(theta_ids):
        frame = cp.asarray(pdata[k].astype('float32'))
        flat  = flat_field(k)
        proj  = bin2(ndimage.gaussian_filter(frame, sigma) / (ndimage.gaussian_filter(flat, sigma) + 1e-5), nb)
        kept  = proj[crop:nz - crop, crop:n - crop]                                  # 1) crop
        S, cf, m = lf_fit(cp.log(kept).get())                                        # 2) fit on the cropped log-division
        kept_c = kept / cp.exp(cp.asarray(S, dtype='float32'))                       # 3) divide: vacuum -> 1
        proj_padded = pad_taper(kept_c, nzobj, nobj, bg=1.0)                         # 4) THEN pad, ramp ends at 1
        p = cp.pad(proj_padded, pad8, 'reflect')                                     # 5) as production
        phase = multiPaganin(p, distance, wavelength, voxelsize, DELTA_BETA, 0.01)[pad8:-pad8, pad8:-pad8]
        phase = fourier_shift_2d(phase, dc=-pos_obj[k, 1], dr=-pos_obj[k, 0])
        rec_lf[i] = phase.get()
        lf_coefs[i], lf_kept[i], lf_ptp[i] = cf, m.mean(), np.ptp(S)
        if i in VIZ_K:
            viz_lf[i] = dict(kept=kept.get(), S=np.exp(S), m=m, kept_c=kept_c.get(), padded=proj_padded.get(), phase=phase.get())
        if i % 200 == 0:
            print(f'  angle {i:5d}  surface ptp {lf_ptp[i]:.4f} (log)   kept {100 * lf_kept[i]:5.1f}% of the cropped frame', flush=True)
cp.get_default_memory_pool().free_all_blocks()

print()
print(f'{len(lf_names)} terms {lf_names}')
print(f'surface ptp (log units): mean {lf_ptp.mean():.4f}, max {lf_ptp.max():.4f};  x delta/beta/2 -> {lf_ptp.mean() * DELTA_BETA / 2:.3f} rad in the phase')
print(f'clip: kept {100 * lf_kept.mean():.1f}% of the cropped frame on average (min {100 * lf_kept.min():.1f}%) -- LOW means the fit is eating the sample')
print(f'lime-box median of the corrected phase at angle 0 (production subtracts global_bg here): {np.median(rec_lf[0, pbox[0]:pbox[1], pbox[2]:pbox[3]]):+.4f} rad;'
      f'  over all angles: mean {np.mean(rec_lf[:, pbox[0]:pbox[1], pbox[2]:pbox[3]]):+.4f}')
print(f'rec_paganin [{rec_paganin.min():.4f}, {rec_paganin.max():.4f}]  ->  rec_lf [{rec_lf.min():.4f}, {rec_lf.max():.4f}]')

In [ ]:
# --- what was fitted and removed, and how it behaves over the scan --------------------------------
lf_show = sorted(viz_lf)
fig, ax = plt.subplots(5, len(lf_show), figsize=(4.4 * len(lf_show), 21), squeeze=False)
for c_, t in enumerate(lf_show):
    v = viz_lf[t]
    for r_, (img, name) in enumerate(((v['kept'], 'cropped division (what is fitted)'),
                                      (v['S'], f"fitted surface exp(S)  (ptp {np.ptp(v['S']):.4f})"),
                                      (v['kept_c'], 'cropped division / surface'),
                                      (v['padded'], 'then padded (ramp ends at 1)'),
                                      (v['phase'], 'Paganin phase'))):
        a = ax[r_][c_]
        lo, hi = np.percentile(img, [1, 99])
        im = a.imshow(img, cmap='gray', vmin=lo, vmax=hi)
        if r_ == 1:
            a.contour(v['m'], [0.5], colors='cyan', linewidths=0.6)                                # pixels surviving the clip
        if r_ >= 3:
            a.add_patch(plt.Rectangle((margin + crop, marginz + crop), kw, kh, fill=False, ec='r', lw=0.8, ls='--'))
        a.set_title(f'{name}\nangle {t} (theta {theta_deg[t]:.1f} deg)', fontsize=9); a.axis('off'); fig.colorbar(im, ax=a, fraction=0.046)
plt.suptitle(f'{TAG}\ncyan = pixels surviving the sigma clip, red dashed = kept data in the padded frame', fontsize=10)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(20, 4))
for j, nm in enumerate(lf_names):
    ax[0].plot(theta_deg, lf_coefs[:, j], lw=0.8, label=nm)
ax[0].set_xlabel('angle (deg)'); ax[0].set_ylabel('coefficient of log(division)')
ax[0].set_title('fitted coefficients vs angle', fontsize=10); ax[0].legend(fontsize=7, ncol=2); ax[0].grid(alpha=0.3)
ax[1].plot(theta_deg, np.exp(lf_coefs[:, 0]), lw=0.8)
ax[1].set_xlabel('angle (deg)'); ax[1].set_ylabel('exp(constant term)')
ax[1].set_title('per-angle vacuum LEVEL (was removed by the fit)', fontsize=10); ax[1].grid(alpha=0.3)
ax[2].plot(theta_deg, lf_kept, lw=0.8)
ax[2].set_xlabel('angle (deg)'); ax[2].set_ylabel('fraction kept by the clip')
ax[2].set_title('how much of the frame is treated as vacuum', fontsize=10); ax[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# --- the actual test: does the reconstruction improve? ---------------------------------------------
lf_sl = slice(nzobj // 2 - 4, nzobj // 2 + 4)
lf_recon = lambda s: cl.fbp(cp.asarray(np.ascontiguousarray(s, dtype='float32')), 'ramp').get()
lf_fbp_before = lf_recon(rec_paganin[:, lf_sl, :])
lf_fbp_after  = lf_recon(rec_lf[:, lf_sl, :])
cp.get_default_memory_pool().free_all_blocks()

lf_yy, lf_xx = np.mgrid[:nobj, :nobj] - (nobj - 1) / 2.0
lf_rr = np.hypot(lf_yy, lf_xx) / (nobj / 2)
lf_edges = np.linspace(0, 1.1, 56); lf_rc = 0.5 * (lf_edges[1:] + lf_edges[:-1])
lf_radial = lambda img: np.array([np.median(img[(lf_rr >= lo) & (lf_rr < hi)]) for lo, hi in zip(lf_edges[:-1], lf_edges[1:])])
lf_p0, lf_p1 = lf_radial(lf_fbp_before[4].real), lf_radial(lf_fbp_after[4].real)
lf_ring = (lf_rc > 0.55) & (lf_rc < 0.75)                         # outside the sample, inside the FBP mask
print(f'vacuum ring 0.55<r<0.75 of the azimuthal median (a flat ring at ~0 is the goal, the sample is NOT in it):')
print(f'   original   mean {lf_p0[lf_ring].mean():+.4f}  ptp {np.ptp(lf_p0[lf_ring]):.4f}')
print(f'   corrected  mean {lf_p1[lf_ring].mean():+.4f}  ptp {np.ptp(lf_p1[lf_ring]):.4f}')
lf_in = lf_rc < 0.75
print(f'(whole r<0.75 peak-to-peak, dominated by the sample, kept only for comparison with the earlier version: '
      f'{np.ptp(lf_p0[lf_in]):.4f} -> {np.ptp(lf_p1[lf_in]):.4f})')

fig, ax = plt.subplots(1, 3, figsize=(19, 5.6))
lo, hi = np.percentile(lf_fbp_before[4].real, [1, 99])
for a, img, t in ((ax[0], lf_fbp_before[4].real, 'FBP, production (rec_paganin)'),
                  (ax[1], lf_fbp_after[4].real, f'FBP, order-{LF_ORDER} surface on the cropped division')):
    im = a.imshow(img, cmap='gray', vmin=lo, vmax=hi)
    a.set_title(t + '  (same grey scale)', fontsize=10); a.axis('off'); fig.colorbar(im, ax=a, fraction=0.046)
ax[2].plot(lf_rc, lf_p0, 'k-', label='production')
ax[2].plot(lf_rc, lf_p1, 'b-', label=f'order-{LF_ORDER} on cropped division')
ax[2].axvspan(0.55, 0.75, color='gray', alpha=0.15)
ax[2].axvline(n / nobj, color='gray', ls='--', lw=0.8); ax[2].axvline(np.sqrt(MASK_R), color='gray', ls=':', lw=0.8)
ax[2].set_xlabel('normalised radius (shaded: vacuum ring, dashed: detector edge, dotted: FBP mask)')
ax[2].set_ylabel('azimuthal median'); ax[2].legend(fontsize=8); ax[2].grid(alpha=0.3)
ax[2].set_title('radial profile of the reconstruction', fontsize=10)
plt.suptitle(TAG, fontsize=10); plt.tight_layout(); plt.show()

## Diagnostic: sweep z1c on the HT side (distance and scale separately)

Both scans took z1c = -17.75 mm from the same nominal number, so neither is checked against the data. The HT flat field
depends on z1c twice. (1) The code-to-sample distance `d_c = (z1 - z1c) * z1 / z1c` (cell 2's `distance_code` reduces to this).
(2) The code's scale in detector pixels: if the HT code plane is at `z`, `code.npy` (made at `Z_NFP`) appears scaled by `z / Z_NFP`
relative to the 1:1 crop (the sample distance z1 cancels there). Rows below are the z1c used for the DISTANCE, columns the z1c used
for the SCALE; the value is the high-passed RMS of the division relative to the baseline at (Z_NFP, Z_NFP).

Read: flat table (variation under ~1%) means z1c is not behind the residue. A minimum on the diagonal is an effective z1c for the HT scan.
Off the diagonal means distance and scale disagree (z1 or the code/probe pair). Needs cells 2-5 and 28 run. Nothing here feeds `rec_paganin`.

In [ ]:
Z_NFP    = -17.75e-3                                      # the z1c the NFP run behind code.npy used
Zs       = Z_NFP + np.linspace(-1.5e-3, 1.5e-3, 7)       # trial z1c, 0.5 mm steps
SWEEP_K  = [0, 1800, 3600, 5400]                          # indices into the 7200 angles
G        = 64

def dist_code(z):                                         # code plane -> sample plane, sample-plane frame
    return (z1 - z) * z1 / z
assert abs(dist_code(z1_code) - distance_code) < 1e-9, (dist_code(z1_code), distance_code)

def flat_z(k, zd, zs):
    """prb x code window; window rescaled as if z1c=zs, propagated as if z1c=zd."""
    dy, dx = int(round(shifts_code[k, 0])), int(round(shifts_code[k, 1]))
    cr, cc = ncode_full // 2 - dy, ncode_full // 2 - dx
    w = code_gpu[cr - nz_full // 2 - G: cr + nz_full // 2 + G, cc - n_full // 2 - G: cc + n_full // 2 + G]
    s = zs / Z_NFP - 1                                    # model(u) = code(c + (u - c)(1 + s)), 1+s = zs/Z_NFP
    if s != 0:
        co = cp.stack(cp.meshgrid((cp.arange(nz_full, dtype='float32') - nz_full / 2) * (1 + s) + nz_full / 2 + G,
                                  (cp.arange(n_full, dtype='float32') - n_full / 2) * (1 + s) + n_full / 2 + G, indexing='ij'))
        w = (ndimage.map_coordinates(w.real.copy(), co, order=3, mode='nearest')
             + 1j * ndimage.map_coordinates(w.imag.copy(), co, order=3, mode='nearest')).astype('complex64')
    else:
        w = w[G:-G, G:-G]
    pc = Propagation(n_full, nz_full, 1, 1, wavelength, voxelsize_full, [dist_code(zd)])
    return cp.abs(cl_prop.D(pc.D(prb_gpu * w, 0), 0)) ** 2

nums = {}
with h5py.File(PATH_H5, 'r') as f:
    for k in SWEEP_K:
        nums[k] = ndimage.gaussian_filter(cp.asarray(f['/exchange/pdata'][k].astype('float32')), sigma)

def hp_rms(k, zd, zs):
    r = hp(bin2(nums[k] / (ndimage.gaussian_filter(flat_z(k, zd, zs), sigma) + 1e-5), nb))
    return float(cp.sqrt(cp.mean(r[crop:-crop, crop:-crop] ** 2)))

base = {k: hp_rms(k, Z_NFP, Z_NFP) for k in SWEEP_K}
E = np.array([[np.mean([hp_rms(k, zd, zs) / base[k] for k in SWEEP_K]) for zs in Zs] for zd in Zs])
print('rows = z1c for DISTANCE, cols = z1c for SCALE (mm); value = hp-RMS / baseline')
print('        ' + ''.join(f'{z*1e3:9.2f}' for z in Zs))
for zd, row in zip(Zs, E):
    print(f'{zd*1e3:7.2f} ' + ''.join(f'{v:9.4f}' for v in row))
i, j = np.unravel_index(E.argmin(), E.shape); d = np.argmin(np.diag(E))
print(f'2D minimum: distance z1c {Zs[i]*1e3:.2f} mm, scale z1c {Zs[j]*1e3:.2f} mm, {E[i, j]:.4f}')
print(f'coupled (diagonal) minimum: z1c {Zs[d]*1e3:.2f} mm, {E[d, d]:.4f}')
cp.get_default_memory_pool().free_all_blocks()

### Same sweep, fine on the SCALE axis

The coarse table saturated: every scale step of 0.5 mm in z1c is a 2.8% stretch (about 30 px at the frame edge), which decorrelates
the code completely (value ~6), so only the column with zero scale change carried information. Scale needs steps near 1e-4
(about 2 um in z1c). Here the DISTANCE stays at `Z_NFP` and only the scale is swept, +-40 um in 4 um steps. The baseline goes through
the same resampling path (a scale of 1e-12 resolves to exact integer coordinates), so the comparison is not biased by interpolation.

In [ ]:
Zs_fine = Z_NFP + np.linspace(-40e-6, 40e-6, 21)
base_rs = {k: hp_rms(k, Z_NFP, Z_NFP * (1 + 1e-12)) for k in SWEEP_K}
EF = np.array([[hp_rms(k, Z_NFP, zs) / base_rs[k] for k in SWEEP_K] for zs in Zs_fine])      # (z, angle)
ef = EF.mean(1)
print(' z1c (mm)    scale s     mean   ' + ''.join(f'  ang {k:5d}' for k in SWEEP_K))
for zs, row, m in zip(Zs_fine, EF, ef):
    print(f'{zs*1e3:9.4f}  {zs / Z_NFP - 1:+.2e}  {m:.4f}  ' + ''.join(f'  {v:9.4f}' for v in row))
j0 = int(ef.argmin()); sl = slice(max(j0 - 2, 0), j0 + 3)
pf = np.polyfit(Zs_fine[sl] * 1e3, ef[sl], 2)
print(f'grid minimum z1c(scale) {Zs_fine[j0]*1e3:.4f} mm ({ef[j0]:.4f});  parabola through 5 points: {-pf[1] / (2 * pf[0]):.4f} mm')
fig, a = plt.subplots(figsize=(7, 4))
a.plot(Zs_fine * 1e3, EF, lw=0.7, alpha=0.6); a.plot(Zs_fine * 1e3, ef, 'k-o', ms=3, label='mean')
a.axvline(Z_NFP * 1e3, color='gray', ls='--', lw=0.8); a.set_xlabel('z1c used for the SCALE (mm)'); a.set_ylabel('hp-RMS / baseline')
a.legend(); a.grid(alpha=0.3); plt.tight_layout(); plt.show()
cp.get_default_memory_pool().free_all_blocks()

## Diagnostic: how much of the vacuum residual is fixed in the detector frame (probe) and how much moves (code)?

The code window moves by hundreds of pixels between angles and the sample rotates, so a structure that STAYS in the same detector pixel in
the division over all angles is the probe (or flat) not matching the beam during the scan. Here the division is built over all angles with the
`CODE_SCALE` model, a per-angle level is taken from the vacuum pixels (pixels whose value barely changes over angle), and the median over
angles of what is left is the fixed map. Reported: the share of the vacuum variance that is fixed, the per-angle level, and the hp-RMS of the
division with and without the fixed map divided out. Reads the notebook's variables; changes nothing in `rec_paganin`.

In [ ]:
kh, kw = nz - 2 * crop, n - 2 * crop
R = np.empty((ntheta, kh, kw), dtype='float32')
with h5py.File(PATH_H5, 'r') as f:
    for i, k in enumerate(theta_ids):
        fr = ndimage.gaussian_filter(cp.asarray(f['/exchange/pdata'][k].astype('float32')), sigma)
        R[i] = bin2(fr / (ndimage.gaussian_filter(flat_field(k), sigma) + 1e-5), nb)[crop:nz - crop, crop:n - crop].get()
cp.get_default_memory_pool().free_all_blocks()
logR = np.log(R)

v = logR.std(0)                                                   # spread over angle, per pixel
vac = v < 3 * np.percentile(v, 10)                                # barely changes over angle -> the sample never covers it
lev = np.median(logR[:, vac], axis=1)                             # per-angle vacuum level (log)
dev = logR - lev[:, None, None]
Pm = np.median(dev, axis=0)                                       # the part fixed in the detector frame

# measured on vacuum pixels, smooth fit under the sample footprint (order 6, where the median is sample-contaminated)
yy, xx = np.mgrid[:kh, :kw]; X = (xx - kw / 2 + .5) / (kw / 2); Y = (yy - kh / 2 + .5) / (kh / 2)
pw = 6
Phi = np.stack([(X ** i * Y ** j).ravel() for i in range(pw + 1) for j in range(pw + 1 - i)], 1)
cf = np.linalg.lstsq(Phi[vac.ravel()], Pm.ravel()[vac.ravel()], rcond=None)[0]
Pfit = (Phi @ cf).reshape(kh, kw)
Pfix = np.where(vac, Pm, Pfit)

tot = dev[:, vac].var(); fixed = Pm[vac].var()
print(f'vacuum pixels: {100 * vac.mean():.0f}% of the cropped frame')
print(f'vacuum variance over (angle, pixel): total {tot:.3e}   fixed in the detector frame {fixed:.3e}   -> {100 * fixed / tot:.0f}% is fixed (probe), {100 - 100 * fixed / tot:.0f}% moves or is noise')
print(f'per-angle level (log): std {lev.std():.4f}; angle 0 is {lev[0] - np.median(lev[1:]):+.4f} from the median of the others')
hp_ = lambda a: float(cp.sqrt(cp.mean(hp(cp.asarray(a)) ** 2)))
before = np.mean([hp_(dev[i]) for i in range(0, ntheta, 8)])
after  = np.mean([hp_(dev[i] - Pfix) for i in range(0, ntheta, 8)])
print(f'hp-RMS of the division (every 8th angle): {before:.5f} -> {after:.5f}  ({100 * (after / before - 1):+.1f}%) with the fixed map divided out')

fig, ax = plt.subplots(1, 4, figsize=(22, 4.6))
for a, img, t in ((ax[0], Pfix, 'fixed map (log), vacuum measured + smooth under the sample'),
                  (ax[1], vac.astype(float), 'vacuum mask'),
                  (ax[2], dev[ntheta // 2] - Pfix, 'one angle after removing it (log)')):
    im = a.imshow(img, cmap='gray', vmin=None if img is vac else np.percentile(img, 1), vmax=None if img is vac else np.percentile(img, 99))
    a.set_title(t, fontsize=9); a.axis('off'); fig.colorbar(im, ax=a, fraction=0.046)
ax[3].plot(theta_deg, lev, lw=0.8); ax[3].set_xlabel('angle (deg)'); ax[3].set_ylabel('vacuum level (log)')
ax[3].set_title('per-angle level', fontsize=9); ax[3].grid(alpha=0.3)
plt.suptitle(TAG, fontsize=10); plt.tight_layout(); plt.show()